In [1]:
''' 
    experiment_data_lists = {
        'Demographic only': demographic_list,
        'Demographic + Behavioral': demographic_list + behavioral_list,
        'Demographic + Behavioral + Psychological': demographic_list + behavioral_list + psychological_list,
        'Behavioral only': behavioral_list,
        'Behavioral + Psychological': behavioral_list + psychological_list,
        'Psychological only': psychological_list
    }
'''

EXPERIMENT_TYPE = 'Demographic + Behavioral + Psychological'

In [4]:
# Standard library imports
import os
import json
import re
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from dotenv import load_dotenv
import os
from openai import OpenAI


from codes.utils import *

In [10]:
file_path = 'Data/Data_SurveyPlusDemographics.txt'
data = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')

# Data preprocess
data, psychological_list = convert_to_natural_language(data)
data, fipi_columns = split_fipi_responses(data)
data.drop(columns=['FIPI_response'], inplace=True, errors='ignore')
#psychological_list.extend(fipi_columns)


train_data = select_experiment_data(data, EXPERIMENT_TYPE)
train_data = process_data_based_on_experiment(train_data, EXPERIMENT_TYPE)

fipi_columns = ['FIPI_1',
 'FIPI_2',
 'FIPI_3',
 'FIPI_4',
 'FIPI_5']
# add labels column
label_columns = fipi_columns
train_data = pd.concat([train_data, data[fipi_columns]], axis=1)

C:\Users\howar\AppData\Local\Temp\ipykernel_21196\1678876330.py:2: DtypeWarning: Columns (115) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')
c:\Users\howar\OneDrive\Documents\GitHub\psych-agent-llm2\codes\utils.py:179: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].apply(clean_numeric)
c:\Users\howar\OneDrive\Documents\GitHub\psych-agent-llm2\codes\utils.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [12]:
train_data['FIPI_5'].describe()

count    8450.000000
mean        5.439882
std         1.389644
min         1.000000
25%         5.000000
50%         6.000000
75%         7.000000
max         7.000000
Name: FIPI_5, dtype: float64

In [13]:
print(Text_SubjectiveLit)
print(Text_TrustPhys)
print(Text_Anxiety)
print(Text_Numeracy)

In a few sentences, please describe to what degree do you feel you have the capacity to obtain, process, and understand basic health information and services needed to make appropriate health decisions?
In a few sentences, please explain the reasons why you trust or distrust your primary care physician. If you do not have a primary care physician, please answer in regard to doctors in general.
In a few sentences, please describe what makes you feel most anxious or worried when visiting the doctor's office.
In a few sentences, please describe an experience in your life that demonstrated your knowledge of health or medical issues.


In [14]:
train_data = train_data.dropna()
# sample number
sample_data = train_data
sample_data = sample_data.dropna()
sample_data = train_data.sample(n=30, random_state=42)
#sample_data = train_data.sample(n=len(train_data), random_state=42)

labels = sample_data[fipi_columns].copy()
labels.reset_index(drop=True, inplace=True)

In [15]:
conditions = [
        "json_fipi_datasets\\all_4.jsonl",
        "json_fipi_datasets\\conditioning_on_all.jsonl",
        "json_fipi_datasets\\holdout_Text_Anxiety.jsonl",
        "json_fipi_datasets\\holdout_Text_Numeracy.jsonl",
        "json_fipi_datasets\\holdout_Text_SubjectiveLit.jsonl",
        "json_fipi_datasets\\holdout_Text_TrustPhys.jsonl",
        
        "json_fipi_datasets\\no_system_prompt.jsonl"
    ]


In [ ]:

load_dotenv()
api_key = os.getenv("API_KEY")

client = OpenAI(api_key = api_key)
def call_gpt(messages):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=200
    ).choices[0].message.content
    return response

In [28]:

label_columns = [fipi_columns]

def load_jsonl(filepath):
    with open(filepath, 'r') as file:
        data = [json.loads(line) for line in file]
    return data

def perform_experiment(condition_file, labels_row, index):
    """
    Enhanced with multi-layer prompting (T0->T1->T2->T3->T4)
    Function signature unchanged, only internal logic enhanced
    """
    json_data = load_jsonl(condition_file)
    generated_responses = {}

    # Get base system messages (T0)
    base_system_messages = json_data[index]['messages']

    questions = {
        "FIPI_1": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Extraverted, enthusiastic (that is, sociable, assertive, talkative, active, NOT reserved, or shy)",
        "FIPI_2": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Agreeable, kind (that is, trusting, generous, sympathetic, cooperative, NOT aggressive, or cold)",
        "FIPI_3": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Dependable, organized (that is, hardworking, responsible, self-disciplined, thorough, NOT careless, or impulsive);",
        "FIPI_4": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Emotionally stable, calm (that is, relaxed, self-confident, NOT anxious, moody, easily upset, or easily stressed)",
        "FIPI_5": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Open to experience, imaginative (that is, curious, reflective, creative, deep, open-minded, NOT conventional)"
    }

    # Multi-layer evolution: T0 -> T1 -> T2 -> T3 -> T4
    current_context = base_system_messages.copy()  # Start with T0
    
    # Define question order for progressive context building
    question_order = ["FIPI_1","FIPI_2","FIPI_3","FIPI_4","FIPI_5"]
    
    # Step 1: Progressive questioning with context accumulation
    for question_key in question_order:
        # Build message with current accumulated context
        messages = current_context.copy()
        messages.append({"role": "user", "content": questions[question_key]})
        
        # Get response
        response = call_gpt(messages)
        generated_responses[question_key] = response
        
        # Accumulate context for next layer (T_i -> T_{i+1})
        current_context.append({"role": "user", "content": questions[question_key]})
        current_context.append({"role": "assistant", "content": response})
        
        # Keep your existing debug output for anxiety
        if question_key == 'Text_Anxiety':
            print(f'this is the message (asking anxiety) for baseline ：{condition_file}')
            print(messages)
            print('Response:')
            print(response)
            print('what we put into the table:')
            print(generated_responses[question_key])

    # Step 2: 比较生成的响应与真实标签，并加入索引 (完全保持不变)
    comparison = {
        "Sample Index": index,
        "Condition": condition_file.split("\\")[-1]
    }
    
    for question in questions.keys():
        comparison[f"{question} Generated"] = generated_responses[question]
        comparison[f"{question} True"] = labels_row[question]

    return comparison

# Baseline 1: zero shot 
def baseline_zero_shot(labels_row, index):
    """
    Enhanced with multi-layer prompting
    Function signature unchanged, only internal logic enhanced
    """
    questions = {
        "FIPI_1": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Extraverted, enthusiastic (that is, sociable, assertive, talkative, active, NOT reserved, or shy)",
        "FIPI_2": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Agreeable, kind (that is, trusting, generous, sympathetic, cooperative, NOT aggressive, or cold)",
        "FIPI_3": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Dependable, organized (that is, hardworking, responsible, self-disciplined, thorough, NOT careless, or impulsive);",
        "FIPI_4": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Emotionally stable, calm (that is, relaxed, self-confident, NOT anxious, moody, easily upset, or easily stressed)",
        "FIPI_5": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Open to experience, imaginative (that is, curious, reflective, creative, deep, open-minded, NOT conventional)"
    }
    generated_responses = {}
    
    # Multi-layer progressive questioning
    current_context = [{"role": "system", "content": 'You are a patient currently visiting a psychologist.'}]
    question_order = ["FIPI_1","FIPI_2","FIPI_3","FIPI_4","FIPI_5"]
    
    for question_key in question_order:
        # Build message with accumulated context
        messages = current_context.copy()
        messages.append({"role": "user", "content": questions[question_key]})
        
        response = call_gpt(messages)
        generated_responses[question_key] = response
        
        # Accumulate context (T_i -> T_{i+1})
        current_context.append({"role": "user", "content": questions[question_key]})
        current_context.append({"role": "assistant", "content": response})
        
        # Keep your existing debug output
        if question_key == 'Text_Anxiety':
            print(f'this is the message (asking anxiety) for baseline ：zero-shot')
            print(messages)
            print('Response:')
            print(response)
            print('what we put into the table:')
            print(generated_responses[question_key])
    
    # Keep original output format unchanged
    comparison = {
        "Sample Index": index,
        "Condition": "Baseline Zero Shot"
    }
    
    for question in questions.keys():
        comparison[f"{question} Generated"] = generated_responses[question]
        comparison[f"{question} True"] = labels_row[question]
    
    return comparison

# Baseline 2: No System Prompt
def baseline_no_system_prompt(condition_file, labels_row, index):
    """
    Enhanced with multi-layer prompting
    Function signature unchanged, only internal logic enhanced
    """
    json_data = load_jsonl(condition_file)
    generated_responses = {}
    
    # Multi-layer progressive questioning starting with minimal system prompt
    current_context = [{"role": "system", "content": 'You are a patient currently visiting a psychologist.'}]
    
    questions = {
        "FIPI_1": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Extraverted, enthusiastic (that is, sociable, assertive, talkative, active, NOT reserved, or shy)",
        "FIPI_2": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Agreeable, kind (that is, trusting, generous, sympathetic, cooperative, NOT aggressive, or cold)",
        "FIPI_3": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Dependable, organized (that is, hardworking, responsible, self-disciplined, thorough, NOT careless, or impulsive);",
        "FIPI_4": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Emotionally stable, calm (that is, relaxed, self-confident, NOT anxious, moody, easily upset, or easily stressed)",
        "FIPI_5": "Please rate the strength of feeling on a scale of 1-7 and only return the number. I see myself as: - Open to experience, imaginative (that is, curious, reflective, creative, deep, open-minded, NOT conventional)"
    }
    
    question_order = ["FIPI_1","FIPI_2","FIPI_3","FIPI_4","FIPI_5"]
    
    # Step 1: Progressive questioning with context accumulation
    for question_key in question_order:
        messages = current_context.copy()
        messages.append({"role": "user", "content": question_key})
        
        response = call_gpt(messages)
        generated_responses[question_key] = response
        
        # Accumulate context (T_i -> T_{i+1})
        current_context.append({"role": "user", "content": question_key})
        current_context.append({"role": "assistant", "content": response})
        

    # Step 2: 比较生成的响应与真实标签，并加入索引 (完全保持不变)
    comparison = {
        "Sample Index": index,
        "Condition": condition_file.split("\\")[-1]
    }
    
    for question in questions.keys():
        comparison[f"{question} Generated"] = generated_responses[question]
        comparison[f"{question} True"] = labels_row[question]
    
    print('this is the comparison for baseline ：no system prompt')
    print(comparison)

    return comparison

In [ ]:
# 模拟多个数据点的实验
def run_all_experiments():
    conditions = [
        "json_fipi_datasets\\all_4.jsonl",
        "json_fipi_datasets\\conditioning_on_all.jsonl",
        "json_fipi_datasets\\holdout_Text_Anxiety.jsonl",
        "json_fipi_datasets\\holdout_Text_Numeracy.jsonl",
        "json_fipi_datasets\\holdout_Text_SubjectiveLit.jsonl",
        "json_fipi_datasets\\holdout_Text_TrustPhys.jsonl",
        
        "json_fipi_datasets\\no_system_prompt.jsonl"
    ]

    all_results = []

    # 遍历所有数据点
    for index, labels_row in labels.iterrows():
        print(f"Running experiments for sample {index}/{len(labels) - 1}...")
        
        # all4 + condition_on_all + holdouts
        for condition in conditions[:-1]:
            comparison = perform_experiment(condition, labels_row, index)
            all_results.append(comparison)

        # Baselines
        baseline_comparison_1 = baseline_zero_shot(labels_row, index)
        all_results.append(baseline_comparison_1)

        baseline_comparison_2 = baseline_no_system_prompt(conditions[-1],labels_row, index)
        all_results.append(baseline_comparison_2)
        results_df = pd.DataFrame(all_results)
        results_df.to_csv("experiment_results_fipi.csv", index=False)

    print("Finished all experiments")
    
    results_df = pd.DataFrame(all_results)
    results_df.to_csv("experiment_results_fipi.csv", index=False)
    print("Results saved to experiment_results_fipi.csv")

# 执行所有实验
run_all_experiments()

Running experiments for sample 0/29...
this is the comparison for baseline ：no system prompt
{'Sample Index': 0, 'Condition': 'no_system_prompt.jsonl', 'FIPI_1 Generated': 'It seems like you might be referencing something specific with "FIPI_1," but I\'m not sure what it is. Could you clarify or provide more details about what you mean by "FIPI_1"? Then I can try to help you better.', 'FIPI_1 True': 2.0, 'FIPI_2 Generated': 'It seems like you\'re referencing terms or codes that I\'m not familiar with. If "FIPI_1" or "FIPI_2" are related to something specific in your life or treatment, could you provide more context? That way, I can better understand how it relates to what you\'re experiencing or discussing with me. If these are personal terms or part of a specific program, sharing a bit more about what they mean to you or how they relate to your current situation might be helpful.', 'FIPI_2 True': 4.0, 'FIPI_3 Generated': "It seems you're continuing with a series of references or codes

KeyboardInterrupt: 

In [31]:
results_df = pd.read_csv('experiment_results_fipi.csv')

print(results_df.head())

   Sample Index                         Condition FIPI_1 Generated  \
0             0                       all_4.jsonl                5   
1             0         conditioning_on_all.jsonl                4   
2             0        holdout_Text_Anxiety.jsonl                5   
3             0       holdout_Text_Numeracy.jsonl                5   
4             0  holdout_Text_SubjectiveLit.jsonl                5   

   FIPI_1 True FIPI_2 Generated  FIPI_2 True FIPI_3 Generated  FIPI_3 True  \
0          2.0                6          4.0                5          6.0   
1          2.0                5          4.0                5          6.0   
2          2.0                6          4.0                5          6.0   
3          2.0                6          4.0                5          6.0   
4          2.0                6          4.0                5          6.0   

  FIPI_4 Generated  FIPI_4 True FIPI_5 Generated  FIPI_5 True  
0                4          4.0               